# LOAN PREDICTIONS

## ✅ PySpark ML Workflow (Step by Step)

A PySpark ML workflow involves loading data, cleaning and splitting it, engineering features, assembling feature vectors, training and tuning a model, evaluating performance, and optionally wrapping everything into a pipeline for production

In [37]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

# For data preprocessing
from pyspark.ml.feature import Imputer,StringIndexer, VectorAssembler, OneHotEncoder

# For the model
from pyspark.ml.classification import DecisionTreeClassifier

# For building the pipeline
from pyspark.ml.pipeline import Pipeline

# For evaluation
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# For hyperparameter tuning
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

In [38]:
spark = SparkSession.builder.appName("loan_predictions").getOrCreate()

### ✅ Visual Summary

Data 
     → Clean 
          → Split

               → Feature Engineering
               → VectorAssembler
               → Model Training
               → (Tuning)
               → Evaluation
               → Pipeline (optional)

#### 1️⃣ Download / Obtain the Dataset

Get the dataset from a source (CSV, Parquet, database, data lake, etc.)

Ensure the schema and target column are understood

#### 2️⃣ Read the Dataset into Spark

- Load the data into a Spark DataFrame
- Infer or define schema explicitly
- Handle headers, delimiters, and null values

✅ Output: Raw Spark DataFrame

In [39]:
data = spark.read.csv("work/dataset/train_loan_prediction.csv", header=True, inferSchema=True  )

#### 3️⃣ Explore and Clean the Data

- Check schema (printSchema)
- Inspect data (show, describe)

Handle:

- missing values
- incorrect data types
- outliers (if needed)


✅ Output: Cleaned DataFrame

In [40]:
data.printSchema()
data.show(5)

root
 |-- Loan_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Married: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- Education: string (nullable = true)
 |-- Self_Employed: string (nullable = true)
 |-- ApplicantIncome: integer (nullable = true)
 |-- CoapplicantIncome: double (nullable = true)
 |-- LoanAmount: integer (nullable = true)
 |-- Loan_Amount_Term: integer (nullable = true)
 |-- Credit_History: integer (nullable = true)
 |-- Property_Area: string (nullable = true)
 |-- Loan_Status: string (nullable = true)

+--------+------+-------+----------+------------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
| Loan_ID|Gender|Married|Dependents|   Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+--------+------+-------+----------+------------+-------------+---------------+-------------

In [41]:
(
    data.select
    (
        [
            F.count(
                F.when(F.col(c).isNull(), 1)
            ).alias(c) for c in data.columns
        ]
    ).show()
)

+-------+------+-------+----------+---------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|Loan_ID|Gender|Married|Dependents|Education|Self_Employed|ApplicantIncome|CoapplicantIncome|LoanAmount|Loan_Amount_Term|Credit_History|Property_Area|Loan_Status|
+-------+------+-------+----------+---------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+
|      0|    13|      3|        15|        0|           32|              0|                0|        22|              14|            50|            0|          0|
+-------+------+-------+----------+---------+-------------+---------------+-----------------+----------+----------------+--------------+-------------+-----------+



#### 4️⃣ Split Data into Training and Test Sets

Split the dataset into:

training data (used for learning)
test data (used only for final evaluation)



Typical split:

70/30 or 80/20

✅ Output:

- train_df
- test_df

In [42]:
train_df, test_df = data.randomSplit([0.8, 0.2], seed=42)

#### 5️⃣ Extract and Prepare Features

Select relevant feature columns

Convert categorical features (if any) using:
- StringIndexer
- OneHotEncoder


Keep numerical features as is or scale them if needed

✅ Output: Feature‑ready columns

In [43]:
pipeline_stages = []

In [44]:
# impute numerical columns with median
numerical_columns_with_null = ['LoanAmount', 'Loan_Amount_Term', 'Credit_History']
imputed_columns = [f'{col}_imputed' for col in numerical_columns_with_null]
# replace nulls with median
imputer = Imputer(inputCols=numerical_columns_with_null, outputCols=imputed_columns, strategy='median')
pipeline_stages.append(imputer)

In [45]:
categorical_columns=['Gender','Married','Dependents','Education','Self_Employed','Property_Area']
for col in categorical_columns:
    # StringIndexer to convert categorical string values to numeric indices
    indexer = StringIndexer(inputCol=col, outputCol=f'{col}_indexed', handleInvalid='keep')
    print(f"Adding StringIndexer for column: {col}")
    pipeline_stages.append(indexer)
    
    # OneHotEncoder to convert indexed categories into one-hot encoded vectors
    encoder = OneHotEncoder(inputCol=indexer.getOutputCol(), outputCol=f'{col}_vec', dropLast=True)
    print(f"Adding OneHotEncoder for column: {col}")
    pipeline_stages.append(encoder)


Adding StringIndexer for column: Gender
Adding OneHotEncoder for column: Gender
Adding StringIndexer for column: Married
Adding OneHotEncoder for column: Married
Adding StringIndexer for column: Dependents
Adding OneHotEncoder for column: Dependents
Adding StringIndexer for column: Education
Adding OneHotEncoder for column: Education
Adding StringIndexer for column: Self_Employed
Adding OneHotEncoder for column: Self_Employed
Adding StringIndexer for column: Property_Area
Adding OneHotEncoder for column: Property_Area


In [46]:
pipeline_stages.append(StringIndexer(inputCol='Loan_Status', outputCol='Loan_Status_indexed'))

#### 6️⃣ Create Feature Vectors

Combine feature columns into a single vector using:``VectorAssembler``

Spark ML requires features in a single vector column (usually named "features").

✅ Output:

- features column
- label column

In [47]:
feature_columns = [f'{col}_vec' for col in categorical_columns] + imputed_columns
feature_columns = feature_columns + ['ApplicantIncome', 'CoapplicantIncome']
feature_columns

['Gender_vec',
 'Married_vec',
 'Dependents_vec',
 'Education_vec',
 'Self_Employed_vec',
 'Property_Area_vec',
 'LoanAmount_imputed',
 'Loan_Amount_Term_imputed',
 'Credit_History_imputed',
 'ApplicantIncome',
 'CoapplicantIncome']

In [48]:
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')
pipeline_stages.append(assembler)

#### 7️⃣ Train the Machine Learning Model

Choose an estimator:

- Classification -> (LogisticRegression, DecisionTree, RandomForest)
- Regression -> (LinearRegression, GBTRegressor)
- Recommendation -> (ALS)

Fit the model using training data

✅ Output: Trained model

In [49]:
dtc = DecisionTreeClassifier(labelCol='Loan_Status_indexed', featuresCol='features')
pipeline_stages.append(dtc)

#### 8️⃣ Create a Pipeline

Combine all steps into a Pipeline:

- feature transformations
- vector assembly
- model training

Ensures:

- clean, reusable workflow
- consistency between training and inference

✅ Output: End‑to‑end ML pipeline

In [50]:
pipeline = Pipeline(stages=pipeline_stages)
model = pipeline.fit(train_df)

#### 9️⃣ Evaluate the Model

Make predictions on the test data

Use an appropriate evaluator:

- Classification → accuracy, AUC, F1
- Regression → RMSE, R²
- Recommendation → RMSE

✅ Output: Model performance metrics

In [51]:
predictions = model.transform(test_df)
evaluator = MulticlassClassificationEvaluator(labelCol='Loan_Status_indexed', predictionCol='prediction', metricName='accuracy')
accuracy = evaluator.evaluate(predictions) * 100
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 76.2887


#### 🔟 (Recommended) Hyperparameter Tuning

Use:

- ParamGridBuilder
- TrainValidationSplit or CrossValidator

Automatically find the best parameter combination

✅ Output: Best trained model

In [52]:
# 1. Create the Grid
paramGrid = (ParamGridBuilder()
    # How many levels deep the tree goes. 
    # 2 is a "baby" tree; 10 is a "wise old" tree.
    .addGrid(dtc.maxDepth, [2, 5, 10])
    
    # How many "bins" the computer uses to sort data.
    # Higher numbers let it see more detail in things like 'Weight'.
    .addGrid(dtc.maxBins, [10, 32, 64])
    
    # The 'Impurity' is the logic used to split a branch.
    # 'Gini' is usually faster; 'Entropy' is a bit more thorough.
    #.addGrid(dtc.impurity, ["gini", "entropy"])
    #.addGrid(dtc.minInstancesPerNode, [1, 5, 10])
    .build())

tvs = (
    TrainValidationSplit(
        estimator=pipeline,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        seed=42
    )
)

best_model = tvs.fit(train_df)
prediction = best_model.transform(test_df)
accuracy = evaluator.evaluate(prediction) * 100
print(f'Best Model Accuracy: {accuracy:.2f}%')

Best Model Accuracy: 78.35%


#### 1️⃣1️⃣ Save and Load the Model (Production)

- Save trained model or pipeline
- Load later for inference or retraining

✅ Output: Production‑ready ML artifact